# 10 — Avaliação final (2025)
Reajuste do XGBoost em 2020–2024 e **uma avaliação única** em 2025. O limiar 0,28 não é modificado em função do teste.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
ROOT=Path.cwd(); OUT=ROOT/"outputs"; DATA=ROOT/"data"/"processed"
SILVER=DATA/"prf_acidentes_silver_2020_2025.parquet"
for p in ["modelos","interpretabilidade","modelo_final"]:
    (OUT/p).mkdir(parents=True,exist_ok=True)
df=pd.read_parquet(SILVER)
df["br_categoria"]=pd.to_numeric(df.br,errors="coerce").astype("Int64").astype("string").fillna("Não informado")
flags=sorted(c for c in df if c.startswith("tracado_") and c!="tracado_via")
categoricas=["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas=["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"]+flags
features=categoricas+numericas
proibidas={"pessoas","mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","classificacao_acidente","acidente_fatal"}
assert set(features).isdisjoint(proibidas)
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
def pre(cat=categoricas,num=numericas):
    try: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse_output=True)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse=True)
    return ColumnTransformer([("categoricas",enc,cat),("numericas",StandardScaler(),num)])
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,fbeta_score,roc_auc_score,average_precision_score,confusion_matrix
def avaliar(y,proba,nome,limiar=0.5):
    pred=(proba>=limiar).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(modelo=nome,threshold=limiar,accuracy=accuracy_score(y,pred),
      precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
      f1=f1_score(y,pred,zero_division=0),f2=fbeta_score(y,pred,beta=2,zero_division=0),
      roc_auc=roc_auc_score(y,proba),pr_auc=average_precision_score(y,proba),
      verdadeiro_negativo=tn,falso_positivo=fp,falso_negativo=fn,verdadeiro_positivo=tp)


In [ ]:
from xgboost import XGBClassifier
treino=df[df.ano<=2024];teste=df[df.ano==2025]
y=treino.acidente_grave
peso=(y==0).sum()/(y==1).sum()
clf=Pipeline([("preprocessamento",pre()),("modelo",XGBClassifier(
    n_estimators=500,learning_rate=.05,max_depth=6,min_child_weight=5,subsample=.8,colsample_bytree=.8,
    scale_pos_weight=peso,objective="binary:logistic",eval_metric="logloss",n_jobs=-1,random_state=42))])
clf.fit(treino[features],y)
proba=clf.predict_proba(teste[features])[:,1]
r=avaliar(teste.acidente_grave,proba,"XGBoost",.28)
display(pd.DataFrame([r]))
pd.DataFrame([{"metrica":k,"valor":v} for k,v in r.items() if k!="modelo"]).to_csv(OUT/"modelo_final"/"resultado_final_2025.csv",index=False)
m=np.array([[r["verdadeiro_negativo"],r["falso_positivo"]],[r["falso_negativo"],r["verdadeiro_positivo"]]])
pd.DataFrame(m,index=["Real não grave","Real grave"],columns=["Previsto não grave","Previsto grave"]).to_csv(OUT/"modelo_final"/"matriz_confusao_2025.csv")
joblib.dump(clf,OUT/"modelo_final"/"xgboost_modelo_final.joblib")
print("Limiar fixado antes do teste: 0.28. Não retreinar nem mudar o limiar a partir do teste.")